# Claim Understanding Agent Demo

This notebook demonstrates the **ClaimUnderstandingAgent** capabilities:

1. **Basic Normalization** - Rule-based text cleanup (no LLM needed)
2. **LLM-based Normalization** - Advanced claim normalization using LLM
3. **Claim Detection** - Detect check-worthy claims from long text

**Requirements:**
- `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` (required for LLM features)
- Basic normalization works without any API keys!


## Setup

Ensure the project is on the Python path.


In [10]:
# Ensure src/ is on the path for this notebook environment
import sys
import pathlib

ROOT = pathlib.Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("✅ Using project root:", ROOT)


✅ Using project root: /Users/ghadir/AFC


In [11]:
# Import required modules
from factcheck_agent.llm_client import get_default_llm_client
from factcheck_agent.models import Claim, DetectedClaim
from factcheck_agent.agents.claim_understanding import ClaimUnderstandingAgent

print("✅ Imports successful!")


✅ Imports successful!


## Part 1: Basic Normalization (No LLM Required)

Basic normalization performs simple rule-based text cleanup:
- Strips whitespace
- Converts to lowercase
- Removes duplicate spaces

This works **without any API keys!**


In [12]:
# Create agent without LLM (basic normalization only)
basic_agent = ClaimUnderstandingAgent(llm=None)

# Test cases
test_claims = [
    "  This is a   TEST claim  ",
    "I heard that like Saudi Arabia became the biggest oil producer??",
    "The   temperature   was   25   degrees   Celsius",
    "  Multiple    spaces    and    mixed    CASE  ",
]

print("=" * 80)
print("Basic Normalization Results")
print("=" * 80)

for i, raw_text in enumerate(test_claims, 1):
    claim = Claim(id=f"basic-{i}", raw_text=raw_text)
    normalized = basic_agent.normalize_claim(claim)
    
    print(f"\n[{i}] Raw:      '{raw_text}'")
    print(f"    Normalized: '{normalized.normalized_text}'")

print("\n" + "=" * 80)


Basic Normalization Results

[1] Raw:      '  This is a   TEST claim  '
    Normalized: 'this is a test claim'

[2] Raw:      'I heard that like Saudi Arabia became the biggest oil producer??'
    Normalized: 'i heard that like saudi arabia became the biggest oil producer??'

[3] Raw:      'The   temperature   was   25   degrees   Celsius'
    Normalized: 'the temperature was 25 degrees celsius'

[4] Raw:      '  Multiple    spaces    and    mixed    CASE  '
    Normalized: 'multiple spaces and mixed case'



## Part 2: LLM-based Normalization (Requires API Key)

Advanced normalization uses LLM to:
- Clean and standardize claim text
- Remove filler words and unnecessary qualifiers
- Normalize entities (dates, numbers, locations)
- Ensure the claim is in a fact-checkable form
- Make claims standalone by replacing pronouns
- Clarify vague terms into explicit language

**Important:** The normalization preserves the claim as stated and does NOT correct facts. When context is provided (e.g., when normalizing detected claims), it uses context only to resolve pronouns, not to change factual content.


In [13]:
# Initialize LLM client
try:
    llm = get_default_llm_client(use_dummy_if_missing_key=False)
    print(f"✅ Using LLM: {type(llm).__name__}\n")
except RuntimeError as e:
    print(f"❌ Error: {e}")
    print("\n⚠️  LLM normalization requires an API key.")
    print("Set OPENAI_API_KEY or ANTHROPIC_API_KEY to use this feature.")
    print("\nSkipping LLM normalization examples...")
    llm = None


✅ Using LLM: OpenAILLMClient



In [14]:
# LLM-based normalization examples
if llm:
    llm_agent = ClaimUnderstandingAgent(llm=llm)
    
    messy_claims = [
        "I heard that like last year Saudi Arabia became the biggest oil producer in the world??",
        "Someone told me that the Earth's temperature increased by like 2 degrees or something",
        "I think maybe the population of China is around 1.4 billion people, but I'm not sure",
        "You know, I read somewhere that NASA said the moon landing was fake?",
    ]
    
    print("=" * 80)
    print("LLM-based Normalization Results")
    print("=" * 80)
    print("\nNote: Normalization preserves claims as stated and does NOT correct facts.\n")
    
    for i, raw_text in enumerate(messy_claims, 1):
        claim = Claim(id=f"llm-{i}", raw_text=raw_text)
        
        print(f"\n[{i}] Raw:      '{raw_text}'")
        print("    Normalizing with LLM...")
        
        normalized = await llm_agent.normalize_claim_llm(claim)
        
        print(f"    Normalized: '{normalized.normalized_text}'")
    
    print("\n" + "=" * 80)
else:
    print("⚠️  Skipping LLM normalization - no API key available")


LLM-based Normalization Results

Note: Normalization preserves claims as stated and does NOT correct facts.


[1] Raw:      'I heard that like last year Saudi Arabia became the biggest oil producer in the world??'
    Normalizing with LLM...
    Normalized: 'In 2023, Saudi Arabia became the largest oil producer in the world.'

[2] Raw:      'Someone told me that the Earth's temperature increased by like 2 degrees or something'
    Normalizing with LLM...
    Normalized: 'The Earth's temperature increased by approximately 2 degrees.'

[3] Raw:      'I think maybe the population of China is around 1.4 billion people, but I'm not sure'
    Normalizing with LLM...
    Normalized: 'The population of China is around 1.4 billion people.'

[4] Raw:      'You know, I read somewhere that NASA said the moon landing was fake?'
    Normalizing with LLM...
    Normalized: 'NASA said the moon landing was fake.'



## Part 3: Claim Detection from Long Text (Requires API Key)

Detect check-worthy claims from articles, posts, or any long text:
- Identifies factual claims worth fact-checking
- Assigns importance scores (0.0-1.0)
- Returns claims with positions in the original text
- **Automatically normalizes detected claims** (both `raw_text` and `normalized_text` are available)
- **Uses original text as context** to resolve pronouns (e.g., "The country" → "Saudi Arabia") without fact-correction


In [16]:
# Claim detection examples
if llm:
    detection_agent = ClaimUnderstandingAgent(llm=llm)
    
    # Example 1: Short article
    sample_text_1 = """
    Saudi Arabia announced plans to invest $100 billion in renewable energy by 2030, 
    according to a statement from the Ministry of Energy. The country, which is currently 
    the world's largest oil producer, aims to diversify its energy sources and reduce 
    carbon emissions. Climate experts have praised the initiative, noting that it represents 
    a significant shift in the country's energy policy. The investment will focus on solar 
    and wind power projects across the kingdom. Some analysts believe this move could 
    transform the global energy market. The announcement came during a climate summit 
    in Riyadh, where officials also revealed plans to achieve net-zero emissions by 2060.
    """
    
    print("=" * 80)
    print("Claim Detection Example 1: Energy Article")
    print("=" * 80)
    print(f"\n📄 Input text ({len(sample_text_1.strip())} characters):")
    print("-" * 80)
    print(sample_text_1.strip())
    print("\n" + "-" * 80)
    print("🔍 Detecting and normalizing claims...")
    print("   (Context from original text is used to resolve pronouns)\n")
    
    # Detect claims (normalization happens automatically by default)
    # The original text is passed as context to help resolve pronouns
    detected_claims_1 = await detection_agent.detect_claims(sample_text_1, min_importance=0.5)
    
    print(f"✅ Detected {len(detected_claims_1)} claim(s):\n")
    for i, claim in enumerate(detected_claims_1, 1):
        print(f"[{i}] Raw Text:      '{claim.raw_text}'")
        if claim.normalized_text:
            print(f"    Normalized:    '{claim.normalized_text}'")
        print(f"    📊 Importance: {claim.importance:.0%}")
        if claim.sentence_index is not None:
            print(f"    📍 Sentence index: {claim.sentence_index}")
        if claim.start_char is not None and claim.end_char is not None:
            print(f"    📍 Position: chars {claim.start_char}-{claim.end_char}")
        print()
    
    if not detected_claims_1:
        print("⚠️  No claims detected above the importance threshold (0.5)")
        print("   Try lowering min_importance or check the input text.\n")
    
    print("=" * 80)
else:
    print("⚠️  Skipping claim detection - no API key available")


Claim Detection Example 1: Energy Article

📄 Input text (709 characters):
--------------------------------------------------------------------------------
Saudi Arabia announced plans to invest $100 billion in renewable energy by 2030, 
    according to a statement from the Ministry of Energy. The country, which is currently 
    the world's largest oil producer, aims to diversify its energy sources and reduce 
    carbon emissions. Climate experts have praised the initiative, noting that it represents 
    a significant shift in the country's energy policy. The investment will focus on solar 
    and wind power projects across the kingdom. Some analysts believe this move could 
    transform the global energy market. The announcement came during a climate summit 
    in Riyadh, where officials also revealed plans to achieve net-zero emissions by 2060.

--------------------------------------------------------------------------------
🔍 Detecting and normalizing claims...
   (Context fro

### How Context-Aware Normalization Works

When detecting claims, the original text is automatically passed as context during normalization. This helps:

1. **Resolve pronouns**: "The country" → "Saudi Arabia" (from context)
2. **Preserve facts**: The claim is kept as stated, not "corrected"
3. **Maintain accuracy**: Even if the LLM knows different facts, it won't change the claim

**Example:** If a claim says "The country is the largest oil producer" and context shows it's about Saudi Arabia, the normalized version will say "Saudi Arabia is the largest oil producer" - NOT "United States is the largest oil producer" (even if that's factually correct).


### Optional: Skip Normalization

You can disable automatic normalization by setting `normalize_detected=False`:


In [17]:
# Example: Detect claims without normalization
if llm:
    sample_text = "Saudi Arabia is the world's largest oil producer. The country plans to invest $100 billion."
    
    print("=" * 80)
    print("Example: Detection Without Normalization")
    print("=" * 80)
    print("\n📄 Input text:", sample_text)
    print("\n🔍 Detecting claims (normalize_detected=False)...\n")
    
    claims_without_norm = await detection_agent.detect_claims(
        sample_text, 
        min_importance=0.5,
        normalize_detected=False
    )
    
    print(f"✅ Detected {len(claims_without_norm)} claim(s) (no normalization):\n")
    for i, claim in enumerate(claims_without_norm, 1):
        print(f"[{i}] Raw Text:      '{claim.raw_text}'")
        print(f"    Normalized:    {claim.normalized_text or '(not normalized)'}")
        print(f"    📊 Importance: {claim.importance:.0%}\n")
    
    print("=" * 80)
    print("\nCompare with normalized version:\n")
    
    claims_with_norm = await detection_agent.detect_claims(
        sample_text, 
        min_importance=0.5,
        normalize_detected=True  # This is the default
    )
    
    print(f"✅ Detected {len(claims_with_norm)} claim(s) (with normalization):\n")
    for i, claim in enumerate(claims_with_norm, 1):
        print(f"[{i}] Raw Text:      '{claim.raw_text}'")
        if claim.normalized_text:
            print(f"    Normalized:    '{claim.normalized_text}'")
        print(f"    📊 Importance: {claim.importance:.0%}\n")
    
    print("=" * 80)


Example: Detection Without Normalization

📄 Input text: Saudi Arabia is the world's largest oil producer. The country plans to invest $100 billion.

🔍 Detecting claims (normalize_detected=False)...

✅ Detected 2 claim(s) (no normalization):

[1] Raw Text:      'Saudi Arabia is the world's largest oil producer'
    Normalized:    (not normalized)
    📊 Importance: 90%

[2] Raw Text:      'The country plans to invest $100 billion'
    Normalized:    (not normalized)
    📊 Importance: 70%


Compare with normalized version:

✅ Detected 2 claim(s) (with normalization):

[1] Raw Text:      'Saudi Arabia is the world's largest oil producer'
    Normalized:    'Saudi Arabia is the world's largest oil producer.'
    📊 Importance: 100%

[2] Raw Text:      'The country plans to invest $100 billion'
    Normalized:    'Saudi Arabia plans to invest $100 billion.'
    📊 Importance: 80%



In [18]:
# Example 2: Longer article with multiple claims
if llm:
    longer_article = """
    In a groundbreaking announcement today, Saudi Arabia revealed plans to invest 
    $100 billion in renewable energy infrastructure by 2030. The initiative, dubbed 
    "Vision 2030 Energy Transformation," aims to position the kingdom as a global leader 
    in clean energy while maintaining its status as the world's largest oil producer.
    
    Energy Minister Prince Abdulaziz bin Salman stated that the investment will create 
    over 200,000 jobs and reduce the country's carbon emissions by 30% within the next 
    decade. The plan includes construction of massive solar farms in the Empty Quarter 
    and wind farms along the Red Sea coast.
    
    International climate experts have praised the move. Dr. Sarah Chen from the 
    International Energy Agency called it "a paradigm shift" that could influence 
    other oil-producing nations. However, some analysts remain skeptical, pointing 
    out that Saudi Arabia still plans to increase oil production capacity to 13 million 
    barrels per day by 2027.
    
    The announcement comes as the kingdom prepares to host the 2030 World Expo, which 
    officials say will be powered entirely by renewable energy. Critics argue that the 
    timeline is overly ambitious, but supporters point to the country's successful 
    completion of the NEOM smart city project as evidence of its capability.
    """
    
    print("\n" + "=" * 80)
    print("Claim Detection Example 2: Longer Article")
    print("=" * 80)
    print(f"\n📄 Input text ({len(longer_article.strip())} characters)")
    print("🔍 Detecting high-importance claims (min_importance=0.6)...\n")
    
    detected_claims_2 = await detection_agent.detect_claims(longer_article, min_importance=0.6)
    
    print(f"✅ Found {len(detected_claims_2)} high-importance claim(s):\n")
    for i, claim in enumerate(detected_claims_2, 1):
        print(f"[{i}] Raw Text:      '{claim.raw_text}'")
        if claim.normalized_text:
            print(f"    Normalized:    '{claim.normalized_text}'")
        print(f"    📊 Importance: {claim.importance:.0%}\n")
    
    print("=" * 80)



Claim Detection Example 2: Longer Article

📄 Input text (1353 characters)
🔍 Detecting high-importance claims (min_importance=0.6)...

✅ Found 12 high-importance claim(s):

[1] Raw Text:      'Saudi Arabia revealed plans to invest $100 billion in renewable energy infrastructure by 2030.'
    Normalized:    'Saudi Arabia plans to invest $100 billion in renewable energy infrastructure by 2030.'
    📊 Importance: 100%

[2] Raw Text:      'Saudi Arabia is the world's largest oil producer.'
    Normalized:    'Saudi Arabia is the world's largest oil producer.'
    📊 Importance: 100%

[3] Raw Text:      'Saudi Arabia plans to increase oil production capacity to 13 million barrels per day by 2027.'
    Normalized:    'Saudi Arabia plans to increase its oil production capacity to 13 million barrels per day by 2027.'
    📊 Importance: 100%

[4] Raw Text:      'Energy Minister Prince Abdulaziz bin Salman stated that the investment will create over 200,000 jobs.'
    Normalized:    'Energy Minist

## Part 4: Interactive Testing

Try your own claims and text below!


In [ ]:
# Test your own claim - Basic Normalization
your_claim = "  Your messy claim text here  "

claim = Claim(id="custom-1", raw_text=your_claim)
normalized = basic_agent.normalize_claim(claim)

print("Raw:      ", your_claim)
print("Normalized:", normalized.normalized_text)


In [ ]:
# Test your own claim - LLM Normalization (if LLM is available)
if llm:
    your_messy_claim = "I heard somewhere that maybe the claim is true??"
    
    claim = Claim(id="custom-2", raw_text=your_messy_claim)
    normalized = await llm_agent.normalize_claim_llm(claim)
    
    print("Raw:      ", your_messy_claim)
    print("Normalized:", normalized.normalized_text)
else:
    print("⚠️  LLM not available - set API key to use this feature")


In [ ]:
# Test your own text - Claim Detection (if LLM is available)
if llm:
    your_text = """
    Your long text here. This could be an article, social media post, 
    or any text containing factual claims that you want to detect.
    """
    
    print("🔍 Detecting and normalizing claims from your text...\n")
    detected = await detection_agent.detect_claims(your_text, min_importance=0.5)
    
    print(f"✅ Detected {len(detected)} claim(s):\n")
    for i, claim in enumerate(detected, 1):
        print(f"[{i}] Raw Text:      '{claim.raw_text}'")
        if claim.normalized_text:
            print(f"    Normalized:    '{claim.normalized_text}'")
        print(f"    📊 Importance: {claim.importance:.0%}\n")
else:
    print("⚠️  LLM not available - set API key to use this feature")


## Summary

The **ClaimUnderstandingAgent** provides three main capabilities:

1. ✅ **Basic Normalization** - Works without any API keys
   - Simple text cleanup and standardization
   - Fast and reliable

2. 🤖 **LLM Normalization** - Requires API key
   - Advanced claim cleaning and normalization
   - Removes filler words, normalizes entities
   - Makes claims fact-checkable
   - **Preserves claims as stated** - does NOT correct facts
   - Optional `context` parameter to resolve pronouns

3. 🔍 **Claim Detection** - Requires API key
   - Detects check-worthy claims from long text
   - Assigns importance scores
   - Returns positioned claims
   - **Automatically normalizes detected claims** (both `raw_text` and `normalized_text` available)
   - **Uses original text as context** to resolve pronouns without fact-correction
   - Use `normalize_detected=False` to skip normalization if needed

**Key Features:**
- When detecting claims, each `DetectedClaim` object includes:
  - `raw_text`: The original detected claim text
  - `normalized_text`: The automatically normalized version (ready for fact-checking)
- **Context-aware normalization**: Original text is used as context to resolve pronouns (e.g., "The country" → "Saudi Arabia") while preserving the claim as stated
- **No fact-correction**: The normalization process preserves claims exactly as stated, only cleaning up language

**Next Steps:**
- Use normalized claims in the fact-checking pipeline
- Process detected claims through the full pipeline
- Integrate with other agents (retrieval, evaluation, etc.)
